# The Text Only Model

This model cannot be feed images or vision data, just text.

![text only model](../showcase_images/text-only-model.png)

In [1]:
import EasyJupyter
import torch.nn as nn
import torch
from model.decoder import Decoder
from model.generator import Generator
from model.RoPE import precompute_freqs_cis
from typing import Optional

In [2]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [3]:
class TextOnlyModel(nn.Module):
    def __init__(self, cfg: BaseConfig):
        """
        The text-only model. This model cannot be feed images or audio, only text.
        """
        super().__init__()
        self.cfg = cfg
        self.decoder = Decoder(cfg)

        self.token_embedding = nn.Embedding(
            num_embeddings=cfg.vocab_size, embedding_dim=cfg.d_model
        )
        self.generator = Generator(cfg)

        # Precompute RoPE frequencies
        self.freqs_cis = precompute_freqs_cis(
            cfg=cfg,
            head_dim=cfg.d_model // cfg.attn_heads,
            context_len=cfg.context_len,
            theta=cfg.pos_embeddings_freq,
        )
        self.initialize_weights()

    def initialize_weights(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        print(
            f"\n\nModel initialized with {sum(p.numel() for p in self.parameters()):,} parameters!\n\n"
        )

    def forward(self, x, combined_mask:torch.Tensor, kv_cache: Optional[list[tuple]] = None):
        """
        Feed the model.
        Args:
            x: The input tensor of shape (batch_size, context_len).
            combined_mask: This mask contains the causal mask which prevents tokens from attending to future tokens, and the document mask which prevents a token in Doc A from attending to tokens in other documents.
                - Paper mentioned: "We use an attention mask that prevents self-attention between different documents within the same sequence. We find that this change had limited impact during in standard pre-training, but find it to be important in continued pre-training on very long sequences."
            kv_caches: Inference only, a list of kv caches.
        """

        # Embed the input tokens
        x_embeds = self.token_embedding(x)

        # Slice the precomputed RoPE frequencies to match the context length, just in case the dataloader yields a smaller batch.
        context_len = x.shape[1]
        batch_freqs_cis = self.freqs_cis[:context_len]

        # Run the decoder
        decoder_out, updated_kv_cache = self.decoder(x_embeds, batch_freqs_cis, combined_mask, kv_cache)
        # Run the generator
        return self.generator(decoder_out)